# 02 — HEDIS / CMS Stars + Care-Gap Prioritisation

End-to-end walkthrough of the **`quality_stars`** bundle introduced in Phase 10.1:

1. **STARS-1** `compute_quality_measures_aggregate` — per-measure HEDIS rates with FY 2025 5/4/3-star CMS benchmark cuts.
2. **STARS-2** `compute_stars_rating_forecast` — per-domain + overall Stars projection at end-of-measurement-year + estimated Quality Bonus Payment dollars.
3. **STARS-3** `compute_care_gap_priority_ranking` — top-N care-gap-closure actions ranked by expected Stars / QBP lift × difficulty.

All three tools are pure-deterministic: same cohort summary → same numeric output, no LLM in the floor.

In [ ]:
import asyncio
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

## Step 1 — Aggregate per-measure HEDIS rates

Real-world MA contract slice. The cohort summary is `{measure_id: {numerator, denominator}}`. The tool maps each measure to its FY 2025 CMS benchmarks and emits the per-measure Stars rating.

In [ ]:
from mcp_server.tools.quality_stars import (
    compute_quality_measures_aggregate,
    compute_stars_rating_forecast,
    compute_care_gap_priority_ranking,
)

cohort_summary = {
    'BCS':       {'numerator': 1400, 'denominator': 1968},   # ~ 0.71
    'COL':       {'numerator': 3300, 'denominator': 5000},   # 0.66
    'CDC-EYE':   {'numerator': 2100, 'denominator': 3000},   # 0.70
    'CDC-HBA1C': {'numerator':  440, 'denominator': 2000},   # 0.22 (poor control — inverted)
    'CBP':       {'numerator': 1200, 'denominator': 1700},   # 0.706
    'MPM-ACE':   {'numerator': 1850, 'denominator': 2000},   # 0.925
    'PCR':       {'numerator':  150, 'denominator': 1500},   # 0.10 (readmits — inverted)
    'SUPD':      {'numerator': 1700, 'denominator': 2000},   # 0.85
}

agg = asyncio.run(compute_quality_measures_aggregate(
    measurement_year=2025,
    cohort_summary=cohort_summary,
    n_eligible_patients=12_000,
))
print(f"Measures computed: {agg.n_measures}")
for m in agg.measures:
    print(f"  {m.measure_id:<12} rate={m.rate:.3f} stars={m.current_stars} "
          f"unmet={m.eligible_unmet_count}")

## Step 2 — Project Stars Rating + estimate QBP

Per-domain (preventive_care + chronic_conditions) + overall projection at end-of-measurement-year. Set `contract_size_thousand_members` to the actual contract size to scale the QBP estimate. (Avalere 2024: 4.0 Stars unlocks ~ 5% benchmark QBP, 4.5+ doubles.)

In [ ]:
fc = asyncio.run(compute_stars_rating_forecast(
    aggregate=agg,
    contract_id='MA-CONTRACT-12345',
    contract_size_thousand_members=50.0,    # 50,000 members
))
print(f"Overall current : {fc.overall_current:.2f} Stars")
print(f"Overall EOM     : {fc.overall_projected_eom:.2f} Stars")
print(f"Estimated QBP   : ${fc.estimated_qbp_dollars_at_overall:,.0f}")
print()
for d in fc.domains:
    print(f"  {d.domain:<22} {d.current_score:.2f} → {d.projected_score_eom:.2f} "
          f"(actions for 4-star: {d.actions_required_for_4_star})")

## Step 3 — Rank care-gap interventions by expected QBP lift

Each action carries a `closure_difficulty` ∈ {easy, moderate, hard, very_hard} and a per-measure `suggested_intervention` (e.g., "Pharmacist-led MTM + insulin titration protocol" for CDC-HBA1C). The ranking is by **expected QBP dollar lift** descending.

In [ ]:
rk = asyncio.run(compute_care_gap_priority_ranking(
    aggregate=agg,
    contract_size_thousand_members=50.0,
    top_n=5,
))
print(f"Top {rk.n_actions} actions — cumulative QBP lift "
      f"${rk.cumulative_expected_qbp_dollars:,.0f}")
for a in rk.actions:
    print(f"\n{a.measure_id} — {a.measure_name}")
    print(f"  Patients with gap : {a.n_eligible_patients_with_gap}")
    print(f"  Stars lift        : +{a.expected_stars_lift:.3f}")
    print(f"  QBP lift          : ${a.expected_qbp_lift_dollars:,.0f}")
    print(f"  Difficulty        : {a.closure_difficulty}")
    print(f"  Intervention      : {a.suggested_intervention}")

## Discussion

- The Stars projection is conservative: it assumes only **30%** of unmet gaps close by EOM. Sensitivity-analyse by re-running with different cohort_summary scenarios.
- The QBP figure linearly scales with `contract_size_thousand_members` — divide by 1,000 to get the per-member dollar impact.
- The same bundle is exposed by the **trustedrisk-quality** specialist on port **8782**, with the A2A v1 FHIR-context extension wired in for cohort-level FHIR access.

See the white paper §9 (`docs/WHITE_PAPER.md`) for the design rationale and the FY 2025 benchmark provenance.